In [1]:
import pandas as pd
import re
from collections import Counter
import os


# Language detection
from langdetect import detect, LangDetectException
from deep_translator import GoogleTranslator

import medspacy


# NLP / Medical NLP
import spacy
import scispacy

I0000 00:00:1787886073.236640    4907 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1787886074.055494    4907 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1787886079.998036    4907 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1787886079.999056    4907 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
W0000 00:0

In [2]:
df = pd.read_csv("../../data/train.csv")
df.head()

,StudyInstanceUID,Report,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10004873229099053869...,Técnica: RMN de la rodilla. Resultados: Rotura...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1.2.826.0.1.3680043.8.498.10004945927472656027...,[DATE]: * MR Knie Rechts 15ch AA Klinische Inl...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1.2.826.0.1.3680043.8.498.10009278692606631573...,Hallazgos:\nNo hay alteraciones en significati...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1.2.826.0.1.3680043.8.498.10009639203170750274...,"In the medial compartment, the meniscus is no...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1.2.826.0.1.3680043.8.498.10013663742400736029...,CONSTATATIONS :\n\nFractures :\nAucune.\n\nAli...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
df['Report']

0       Técnica: RMN de la rodilla. Resultados: Rotura...
1       [DATE]: * MR Knie Rechts 15ch AA Klinische Inl...
2       Hallazgos:\nNo hay alteraciones en significati...
3        In the medial compartment, the meniscus is no...
4       CONSTATATIONS :\n\nFractures :\nAucune.\n\nAli...
                              ...                        
4402    Exam Type: MRI KNEE RIGHT WO CONTRAST\nExam Da...
4403    [DATE]: *MR Knie Rechts 15ch AA Klinische Inli...
4404    SAĞ DİZ MRG. Tetkik protokolü: Çok düzlemli, ç...
4405    MRI of Knee with \n-Locator, SG PD FatSat, SG ...
4406    Técnica: RMN de la rodilla. Resultados: Rotura...
Name: Report, Length: 4407, dtype: object

**Language Detection**

In [4]:
def detect_language(text):
    if not isinstance(text, str) or not text.strip():
        return "unknown"

    try:
        return detect(text)
    except LangDetectException:
        return "unknown"

In [5]:
df["ReportLanguage"] = df["Report"].apply(detect_language)

In [6]:
df[["Report", "ReportLanguage"]].head()

,Report,ReportLanguage
0,Técnica: RMN de la rodilla. Resultados: Rotura...,es
1,[DATE]: * MR Knie Rechts 15ch AA Klinische Inl...,nl
2,Hallazgos:\nNo hay alteraciones en significati...,es
3,"In the medial compartment, the meniscus is no...",en
4,CONSTATATIONS :\n\nFractures :\nAucune.\n\nAli...,fr


In [7]:
df["ReportLanguage"].value_counts()


ReportLanguage
en    1736
es     681
tr     546
hr     406
el     321
de     262
bg     220
nl     153
fr      81
it       1
Name: count, dtype: int64

**Translation (finding terms)**

In [8]:
excluded_translation_terms = {"DATE"}

In [9]:
def find_abbreviations(text):
    pattern = r'\b[A-Z]{2,}(?:[-/][A-Z0-9]+)*\b'
    
    terms = re.findall(pattern, text)
    
    return [
        term for term in terms
        if term not in excluded_translation_terms
    ]

In [10]:
text = df["Report"].iloc[1]

print(text)
print(find_abbreviations(text))

all_abbreviations = Counter()

for text in df["Report"].dropna():
    all_abbreviations.update(find_abbreviations(text))

[DATE]: * MR Knie Rechts 15ch AA Klinische Inlichtingen: [DATE]. Diagnostische vraagstellling: Meniscusscheur/mediaal? Scanprotocol (DRB) : sag intermediair gewogen seq zonder en met fs, ax/ cor pd gewogen seq fs, cor T1 gewogen seq Bevindingen:
['MR', 'AA', 'DRB']


In [11]:
print(all_abbreviations.most_common(30))

[('ACL', 1729), ('MRI', 1526), ('MCL', 1447), ('PCL', 1303), ('COMPARTMENT', 1076), ('LCL', 940), ('LIGAMENTS', 694), ('FS', 693), ('II', 681), ('FINDINGS', 556), ('OA', 467), ('KNEE', 445), ('TT-TG', 440), ('IMPRESSION', 430), ('MPFL', 395), ('MR', 382), ('TECHNIQUE', 378), ('MRG', 370), ('LATERAL', 370), ('MEDIAL', 368), ('SG', 363), ('TIME', 362), ('COLLATERAL', 358), ('CRUCIATE', 357), ('III', 356), ('CONTRAST', 355), ('EXTENSOR', 353), ('WO', 352), ('JOINT', 351), ('PATELLOFEMORAL', 351)]


**Interpret abbreviation context**

In [12]:
for abbreviation in ["TT-TG", "MR", "MRG", "SG", "WO"]:
    print(f"\n--- {abbreviation} ---")
    
    for text in df["Report"].dropna():
        if abbreviation in text:
            print(text[:500])
            break


--- TT-TG ---
COMPARISON:None.

CLINICAL HISTORY:Assess proximal patellar tendon tear left knee.

PROCEDURE:Multisequence, multiplanar unenhanced MRI of the left knee according to routine protocol.

IMAGING FINDINGS:

Nondiagnostic scout localizing images reviewed..

The lateral meniscus is intact. Inferiorly surfacing oblique tear posterior horn medial meniscus extending into the body, which is peripherally extruded. Multilobulated para meniscal cyst measuring 6 x 2 x 2 mm noted adjacent to the posterior hor

--- MR ---
[DATE]: * MR Knie Rechts 15ch AA Klinische Inlichtingen: [DATE]. Diagnostische vraagstellling: Meniscusscheur/mediaal? Scanprotocol (DRB) : sag intermediair gewogen seq zonder en met fs, ax/ cor pd gewogen seq fs, cor T1 gewogen seq Bevindingen:

--- MRG ---
SOL DİZ MRG. Tetkik protokolü: Çok düzlemli, çok sekanslı. Bulgular: Lateral menisküs ve medial menisküs normaldir. Ön çapraz bağ ve arka çapraz bağ, lateral ve medial kollateral ligaman normaldir.Distal kuadrisep

In [13]:
for abbreviation, count in all_abbreviations.most_common():
    print(abbreviation, count)

ACL 1729
MRI 1526
MCL 1447
PCL 1303
COMPARTMENT 1076
LCL 940
LIGAMENTS 694
FS 693
II 681
FINDINGS 556
OA 467
KNEE 445
TT-TG 440
IMPRESSION 430
MPFL 395
MR 382
TECHNIQUE 378
MRG 370
LATERAL 370
MEDIAL 368
SG 363
TIME 362
COLLATERAL 358
CRUCIATE 357
III 356
CONTRAST 355
EXTENSOR 353
WO 352
JOINT 351
PATELLOFEMORAL 351
MECHANISM 348
MUSCLES/TENDONS 347
SOFT 347
TISSUES 347
OSSEOUS 346
STRUCTURES 346
SPACE 346
POSTEROLATERAL 346
CORNER 346
RMN 333
PD 316
RIGHT 231
LEFT 219
SOL 177
LCA 161
PF 158
AA 146
SI 139
AX 133
CONCLUSION 123
ICRS 123
CO 121
IV 118
REDACTED 110
FT 99
ID 94
FCL 87
COMPARISON 87
IMAGING 85
INDICATION 84
CLINICAL 81
LCP 65
VKB 63
LCM 60
INTERPRETATION 59
FP 55
IT 55
DD 55
HISTORY 53
PROCEDURE 52
HKB 51
CONCLUSIE 48
CC 39
AND 38
NAME 38
PRIOR 34
TECHNICAL 34
INFORMATION 34
PROTOCOL 34
BILATERAL 28
TEXNIKH 28
NOTE 27
CONSTATATIONS 24
AP 24
NC/NA 23
DWI 20
LIGAMENT 20
PDFS 19
II-III 17
YEAR 14
STIR 14
TG 14
DRB 12
MENISCUS 12
VU 12
NO 11
ANTERIOR 11
TTTG 11
PVNS 10
TF 10
CO

**Dictionary**

In [258]:
# MEDICAL DICTIONARY

medical_dictionary = {

    # ==========================================================
    # LIGAMENTS
    # ==========================================================

    "anterior cruciate ligament": "ACL",
    "anterior cruciate ligament (ACL)": "ACL",
    "ACL": "ACL",

    "anterior cruciate ligament injury": "ACL injury",
    "ACL injury": "ACL injury",
    "ACL sprain": "ACL sprain",
    "grade 1 injury of ACL": "grade 1 ACL injury",
    "grade 1 ACL injury": "grade 1 ACL injury",
    "grade I injury of ACL": "grade 1 ACL injury",
    "grade 1 injury or degeneration of ACL":
        "grade 1 ACL injury or degeneration",
    "partial tear of ACL": "partial ACL tear",
    "ACL partial tear": "partial ACL tear",
    "tear of ACL": "ACL tear",
    "ACL tear": "ACL tear",
    "subtotal ACL tear": "subtotal ACL tear",
    "ACL mucinous degeneration": "ACL mucoid degeneration",
    "ACL myxoid degeneration": "ACL mucoid degeneration",
    "ACL mucoid degeneration": "ACL mucoid degeneration",

    # Greek - ACL
    "πρόσθιος χιαστός σύνδεσμος": "ACL",
    "πρόσθιου χιαστού συνδέσμου": "ACL",
    "πρόσθιος χιαστός": "ACL",
    "πρόσθιου χιαστού": "ACL",

    # Turkish - ACL
    "ön çapraz bağ": "ACL",
    "ön çapraz bağda": "ACL",
    "ön çapraz bağın": "ACL",

    # ----------------------------------------------------------

    "posterior cruciate ligament": "PCL",
    "posterior cruciate ligament (PCL)": "PCL",
    "PCL": "PCL",

    "PCL injury": "PCL injury",
    "PCL sprain": "PCL sprain",
    "grade 1 injury of PCL": "grade 1 PCL injury",
    "grade I injury of PCL": "grade 1 PCL injury",
    "grade 1 PCL injury": "grade 1 PCL injury",
    "partial tear of PCL": "partial PCL tear",
    "partial tear of the PCL": "partial PCL tear",
    "PCL partial tear": "partial PCL tear",
    "severe partial tear of PCL": "severe partial PCL tear",
    "PCL avulsion fracture": "PCL avulsion fracture",
    "avulsion fracture of PCL": "PCL avulsion fracture",

    # Greek - PCL
    "οπίσθιος χιαστός σύνδεσμος": "PCL",
    "οπίσθιου χιαστού συνδέσμου": "PCL",
    "οπίσθιος χιαστός": "PCL",
    "οπίσθιου χιαστού": "PCL",

    # Turkish - PCL
    "arka çapraz bağ": "PCL",
    "arka çapraz bağda": "PCL",

    # ----------------------------------------------------------

    "medial collateral ligament": "MCL",
    "medial collateral ligament (MCL)": "MCL",
    "MCL": "MCL",

    "MCL sprain": "MCL sprain",
    "grade I sprain of MCL": "grade 1 MCL sprain",
    "grade II sprain of MCL": "grade 2 MCL sprain",
    "partial tear of MCL": "partial MCL tear",
    "MCL partial tear": "partial MCL tear",
    "MCL injury": "MCL injury",
    "MCL tear": "MCL tear",

    # Spanish
    "ligamento colateral medial": "MCL",
    "ligamento colateral interno": "MCL",
    "LCM": "MCL",
    "lesión LCM": "MCL injury",

    # Turkish
    "medial kollateral ligaman": "MCL",
    "medial kollateral ligamanında": "MCL",
    "medial kollateral ligaman çevresinde": "MCL",

    # ----------------------------------------------------------

    "lateral collateral ligament": "LCL",
    "lateral collateral ligament (LCL)": "LCL",
    "LCL": "LCL",

    "LCL injury": "LCL injury",
    "LCL sprain": "LCL sprain",
    "LCL tear": "LCL tear",

    # ==========================================================
    # MENISCI
    # ==========================================================

    "medial meniscus": "medial meniscus",
    "medial meniscus tear": "medial meniscus tear",
    "tear of medial meniscus": "medial meniscus tear",

    "posterior horn of medial meniscus":
        "posterior horn of medial meniscus",

    "posterior horn of medial meniscus tear":
        "posterior horn medial meniscus tear",

    "medial meniscus posterior horn tear":
        "posterior horn medial meniscus tear",

    "medial meniscus degeneration":
        "medial meniscus degeneration",

    "grade 1 medial meniscus degeneration":
        "grade 1 medial meniscus degeneration",

    "grade 2 medial meniscus degeneration":
        "grade 2 medial meniscus degeneration",

    "grade 3 medial meniscus degeneration":
        "grade 3 medial meniscus degeneration",

    "medial meniscus extrusion":
        "medial meniscus extrusion",

    "medial meniscus degeneration with extrusion":
        "medial meniscus degeneration with extrusion",

    # ----------------------------------------------------------

    "lateral meniscus": "lateral meniscus",
    "lateral meniscus tear": "lateral meniscus tear",
    "tear of lateral meniscus": "lateral meniscus tear",

    "anterior horn of lateral meniscus":
        "anterior horn of lateral meniscus",

    "anterior horn of lateral meniscus tear":
        "anterior horn lateral meniscus tear",

    "posterior horn of lateral meniscus":
        "posterior horn of lateral meniscus",

    "posterior horn of lateral meniscus tear":
        "posterior horn lateral meniscus tear",

    "lateral meniscus degeneration":
        "lateral meniscus degeneration",

    "grade 1 lateral meniscus degeneration":
        "grade 1 lateral meniscus degeneration",

    "grade 2 lateral meniscus degeneration":
        "grade 2 lateral meniscus degeneration",

    "grade 3 lateral meniscus degeneration":
        "grade 3 lateral meniscus degeneration",

    # ----------------------------------------------------------

    "meniscal tear": "meniscus tear",
    "meniscus tear": "meniscus tear",
    "meniscus degeneration": "meniscus degeneration",
    "meniscal degeneration": "meniscus degeneration",

    "meniscopathy": "meniscus degeneration",
    "medial meniscopathy": "medial meniscus degeneration",
    "lateral meniscopathy": "lateral meniscus degeneration",

    "grade 2 meniscopathy":
        "grade 2 meniscus degeneration",

    "grade 3 meniscopathy":
        "grade 3 medial meniscus degeneration",

    "grade 2 intrasubstance signal":
        "grade 2 meniscal intrasubstance degeneration",

    "grade 2 intrasubstance signal of posterior horn of medial meniscus":
        "grade 2 medial meniscus degeneration",

    # Turkish
    "medial menisküs": "medial meniscus",
    "medial menisküste": "medial meniscus",
    "lateral menisküs": "lateral meniscus",
    "lateral menisküste": "lateral meniscus",
    "meniskopati": "meniscus degeneration",
    "medial meniskopati": "medial meniscus degeneration",

    # Croatian / Serbian
    "medijalni menisk": "medial meniscus",
    "medijalnog meniska": "medial meniscus",
    "lateralni menisk": "lateral meniscus",
    "lateralnog meniska": "lateral meniscus",
    "ruptura medijalnog meniska": "medial meniscus tear",
    "degeneracija medijalnog meniska":
        "medial meniscus degeneration",

    # ==========================================================
    # FRACTURES / BONE INJURIES
    # ==========================================================

    "tibial plateau fracture": "tibial plateau fracture",
    "fracture of tibial plateau": "tibial plateau fracture",
    "split fracture of tibial plateau":
        "split tibial plateau fracture",

    "tibial intercondylar eminence fracture":
        "tibial intercondylar eminence fracture",

    "intercondylar eminence fracture":
        "tibial intercondylar eminence fracture",

    "posterior tibial intercondylar eminence fracture":
        "posterior tibial intercondylar eminence fracture",

    "bone contusion": "bone contusion",
    "bone bruise": "bone contusion",

    "bone marrow edema": "bone marrow edema",
    "bone marrow oedema": "bone marrow edema",
    "bone edema": "bone marrow edema",

    "osteochondral lesion": "osteochondral lesion",
    "osteochondral injury": "osteochondral injury",

    "Segond fracture": "Segond fracture",
    "Segond's fracture": "Segond fracture",

    "osteonecrosis": "osteonecrosis",
    "small osteonecrosis": "small osteonecrosis",

    # ==========================================================
    # CARTILAGE / OSTEOARTHRITIS
    # ==========================================================

    "osteoarthritis": "osteoarthritis",
    "osteoarthritis of knee": "knee osteoarthritis",
    "knee osteoarthritis": "knee osteoarthritis",

    "gonarthrosis": "knee osteoarthritis",
    "gonartroz": "knee osteoarthritis",

    # ----------------------------------------------------------

    "chondromalacia": "chondromalacia",
    "grade 1 chondromalacia": "grade 1 chondromalacia",
    "grade 2 chondromalacia": "grade 2 chondromalacia",
    "grade 3 chondromalacia": "grade 3 chondromalacia",
    "grade 4 chondromalacia": "grade 4 chondromalacia",
    "grade 3-4 chondromalacia": "grade 3-4 chondromalacia",

    "patellar chondromalacia": "patellar chondromalacia",
    "patellofemoral chondromalacia":
        "patellofemoral chondromalacia",

    "cartilage degeneration": "cartilage degeneration",
    "cartilage thinning": "cartilage thinning",

    "full-thickness cartilage defect":
        "full-thickness cartilage defect",

    "cartilage defect": "cartilage defect",
    "cartilage damage": "cartilage damage",
    "cartilage loss": "cartilage loss",

    # Turkish / other languages
    "hondromalazi": "chondromalacia",
    "hondromalacia": "chondromalacia",
    "hondromalacija": "chondromalacia",

    # ==========================================================
    # SYNOVIUM / FLUID / BURSA
    # ==========================================================

    "synovitis": "synovitis",
    "knee synovitis": "knee synovitis",
    "synovial hypertrophy": "synovial hypertrophy",

    # ----------------------------------------------------------

    "joint effusion": "joint effusion",
    "knee joint effusion": "knee joint effusion",
    "minimal joint effusion": "minimal joint effusion",
    "mild joint effusion": "mild joint effusion",

    "small joint effusion": "joint effusion",
    "moderate joint effusion": "joint effusion",
    "large joint effusion": "joint effusion",

    # Variants found in reports
    "minimal fluid increase": "joint effusion",
    "mild fluid increase": "joint effusion",
    "slight fluid increase": "joint effusion",
    "increased fluid": "joint effusion",
    "fluid increase": "joint effusion",
    "increased amount of fluid": "joint effusion",
    "increased amount of intra-articular fluid":
        "joint effusion",

    # ----------------------------------------------------------

    "suprapatellar bursitis": "suprapatellar bursitis",
    "deep infrapatellar bursitis":
        "deep infrapatellar bursitis",
    "infrapatellar bursitis": "infrapatellar bursitis",
    "anserine bursitis": "anserine bursitis",
    "pes anserine bursitis": "pes anserine bursitis",

    "patellar plica": "patellar plica",

    "lipoma arborescens": "lipoma arborescens",
    "lipohemarthrosis": "lipohemarthrosis",

    # ==========================================================
    # CYSTS
    # ==========================================================

    "Baker's cyst": "Baker's cyst",
    "Baker cyst": "Baker's cyst",
    "popliteal cyst": "Baker's cyst",
    "popliteal cystic lesion": "Baker's cyst",

    "ganglion cyst": "ganglion cyst",
    "ACL ganglion cyst": "ACL ganglion cyst",
    "PCL ganglion cyst": "PCL ganglion cyst",
    "ligament ganglion cyst": "ligament ganglion cyst",

    "parameniscal cyst": "parameniscal cyst",
    "parameniscal cystic lesion": "parameniscal cyst",

    "synovial cyst": "synovial cyst",

    # ==========================================================
    # TENDONS
    # ==========================================================

    "patellar tendinosis": "patellar tendinosis",
    "patellar tendon tendinosis":
        "patellar tendinosis",

    "quadriceps tendinosis": "quadriceps tendinosis",
    "quadriceps tendon tendinosis":
        "quadriceps tendinosis",

    "patellar tendon injury": "patellar tendon injury",
    "quadriceps tendon injury": "quadriceps tendon injury",

    "tendinopathy": "tendinopathy",
    "tendinosis": "tendinosis",

    "grade 1 tendon injury": "grade 1 tendon injury",

    "rupture of fibers of the patellar tendon":
        "patellar tendon rupture",

    "rupture of the patellar tendon":
        "patellar tendon rupture",

    "patellar tendon rupture":
        "patellar tendon rupture",

    # ==========================================================
    # SOFT TISSUE
    # ==========================================================

    "soft tissue edema": "soft tissue edema",
    "subcutaneous edema": "subcutaneous edema",
    "muscle edema": "muscle edema",

    "subcutaneous and muscle edema":
        "subcutaneous and muscle edema",

    "hematoma": "hematoma",
    "subcutaneous hematoma": "subcutaneous hematoma",
    "prepatellar hematoma": "prepatellar hematoma",

    # ==========================================================
    # PATELLA / PATELLOFEMORAL
    # ==========================================================

    "patellar maltracking": "patellar maltracking",
    "patella maltracking": "patellar maltracking",

    "patellar subluxation": "patellar subluxation",
    "patellar dislocation": "patellar dislocation",

    "patella alta": "patella alta",

    "Wiberg type I patella": "Wiberg type I patella",
    "Wiberg type II patella": "Wiberg type II patella",
    "Wiberg II": "Wiberg type II patella",

    "bipartite patella": "bipartite patella",

    "patellofemoral syndrome": "patellofemoral syndrome",

    # ==========================================================
    # OTHER LESIONS
    # ==========================================================

    "osteochondroma": "osteochondroma",
    "exostosis": "exostosis",

    "osteochondral loose body":
        "osteochondral loose body",

    "loose body": "loose body",

    "osteochondral foreign body":
        "osteochondral loose body",

    "intraosseous hemangioma":
        "intraosseous hemangioma",

    "nodular fasciitis": "nodular fasciitis",

    "tenosynovial giant cell tumor":
        "tenosynovial giant cell tumor",

    "fibroma of the tendon sheath":
        "fibroma of the tendon sheath",

    "tenosynovial fibroma":
        "tenosynovial fibroma",

    # ==========================================================
    # ANATOMICAL / OTHER RECURRING CONDITIONS
    # ==========================================================

    "IT band syndrome": "iliotibial band syndrome",
    "iliotibial band syndrome":
        "iliotibial band syndrome",

    "iliotibial band impingement":
        "iliotibial band impingement",

    "pes anserine paratendinitis":
        "pes anserine paratendinitis",

    "patellar tendon impingement":
        "patellar tendon impingement",

    # ==========================================================
    # HOFFA FAT PAD
    # ==========================================================

    "hoffa fat pad impingement":
        "Hoffa Fat Pad Impingement",

    "hoffa fat pad friction syndrome":
        "Hoffa Fat Pad Impingement",

    "hoffa fat pad clamping":
        "Hoffa Fat Pad Impingement",

    # ==========================================================
    # MRI
    # ==========================================================

    "magnetic resonance imaging": "MRI",
    "magnetic resonance imaging (MRI)": "MRI",
    "MRI": "MRI",

    "resonancia magnética": "MRI",
    "resonancia magnética nuclear": "MRI",

    # ==========================================================
    # MPFL
    # ==========================================================

    "medial patellofemoral ligament": "MPFL",
    "medial patellofemoral ligament (MPFL)": "MPFL",
    "MPFL": "MPFL",

    "MPFL injury": "MPFL injury",
    "MPFL tear": "MPFL tear",
    "partial tear of MPFL": "partial MPFL tear",
    "MPFL partial tear": "partial MPFL tear",

    "grade 1 injury of MPFL":
        "grade 1 MPFL injury",

    "grade 1 MPFL injury":
        "grade 1 MPFL injury",

    # ==========================================================
    # OA
    # ==========================================================

    "OA": "osteoarthritis",

    # ==========================================================
    # FS
    # ==========================================================

    "FS": "FS",
}

In [15]:
# TERM PROTECTION

def protect_terms(text, glossary, technical_terms=None):
    """
    Protects medical and technical terms before translation.

    Longer expressions are processed first to prevent partial matches.

    Medical acronyms are matched case-sensitively.

    Medical phrases are matched case-insensitively.

    Technical MRI terms are also protected when provided.

    Terms marked for review are detected but not replaced.
    """

    if technical_terms is None:
        technical_terms = set()

    placeholders = {}
    flagged_terms = []

    # Combine medical and technical terms

    all_terms = set(glossary.keys()) | set(technical_terms)

    # Process longer expressions first

    sorted_terms = sorted(
        all_terms,
        key=len,
        reverse=True
    )

    for i, term in enumerate(sorted_terms):

        # Determine replacement

        if term in glossary:
            target = glossary[term]
        else:
            target = term

        # Build matching pattern

        if term in SHORT_ACRONYMS or term in technical_terms:

            pattern = re.compile(
                r'(?<!\w)' +
                re.escape(term) +
                r'(?!\w)'
            )

        else:

            pattern = re.compile(
                r'(?<!\w)' +
                re.escape(term) +
                r'(?!\w)',
                re.IGNORECASE
            )

        # Handle terms requiring review

        if term in FLAG_FOR_REVIEW:

            if pattern.search(text):
                flagged_terms.append(term)

            continue

        # Protect the term

        if pattern.search(text):

            token = f"XTERM{i}X"

            text = pattern.sub(
                token,
                text
            )

            placeholders[token] = target

    return text, placeholders, flagged_terms

**Validation 1**

In [16]:

# MEDICAL CONTENT VALIDATION

def validate_medical_content(text):
    """
    Performs basic validation of the translated medical report.

    This validation checks for empty text and obvious translation
    or placeholder artifacts. It does not determine clinical correctness.
    """

    warnings = []

    # Suspicious translation artifacts

    suspicious_patterns = [
        "XTERM",
        "undefined",
        "null",
        "None"
    ]

    for pattern in suspicious_patterns:

        if pattern.lower() in text.lower():

            warnings.append(
                f"Suspicious text detected: {pattern}"
            )

    # Empty text

    if not isinstance(text, str) or not text.strip():

        warnings.append(
            "Medical text is empty."
        )

    return {
        "is_valid": len(warnings) == 0,
        "warnings": warnings
    }

**Labels**

In [17]:
LABELS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture"
]

In [18]:
# NLP LABEL EXTRACTION

def extract_nlp_labels(text):
    """
    Converts the normalized medical report
    into the 12 dataset labels.
    """

    labels = {label: 0 for label in LABELS}

    return labels

In [19]:
# MEDICAL FINDINGS TO DATASET LABELS

LABEL_PATTERNS = {

    "ACL": [
        r"\bACL\b",
        r"\banterior cruciate ligament\b",
        r"\bACL tear\b",
        r"\bACL injury\b",
        r"\bACL sprain\b",
        r"\bACL mucoid degeneration\b",
        r"\bACL ganglion cyst\b",
    ],

    "MCL": [
        r"\bMCL\b",
        r"\bmedial collateral ligament\b",
        r"\bMCL tear\b",
        r"\bMCL injury\b",
        r"\bMCL sprain\b",
    ],

    "Medial Meniscus": [
        r"\bmedial meniscus\b",
        r"\bmedial meniscus tear\b",
        r"\bmedial meniscus degeneration\b",
        r"\bmedial meniscopathy\b",
        r"\bposterior horn of medial meniscus\b",
    ],

    "Lateral Meniscus": [
        r"\blateral meniscus\b",
        r"\blateral meniscus tear\b",
        r"\blateral meniscus degeneration\b",
        r"\banterior horn of lateral meniscus\b",
        r"\bposterior horn of lateral meniscus\b",
    ],

    "Medial OA": [
        r"\bmedial compartment osteoarthritis\b",
        r"\bmedial compartment OA\b",
        r"\bmedial femorotibial osteoarthritis\b",
    ],

    "Lateral OA": [
        r"\blateral compartment osteoarthritis\b",
        r"\blateral compartment OA\b",
        r"\blateral femorotibial osteoarthritis\b",
    ],

    "PF OA": [
        r"\bpatellofemoral osteoarthritis\b",
        r"\bpatellofemoral OA\b",
        r"\bpatellofemoral chondromalacia\b",
        r"\bpatellar chondromalacia\b",
    ],

    "Effusion": [
        r"\bjoint effusion\b",
        r"\bknee joint effusion\b",
        r"\beffusion\b",
        r"\bhemarthrosis\b",
    ],

    "Synovitis": [
        r"\bsynovitis\b",
        r"\bsynovial hypertrophy\b",
    ],

    "Baker's": [
        r"\bBaker's cyst\b",
        r"\bBaker cyst\b",
        r"\bpopliteal cyst\b",
    ],

    "Contusion": [
        r"\bbone contusion\b",
        r"\bbone bruise\b",
        r"\bmuscle contusion\b",
        r"\bcontusion\b",
    ],

    "Fracture": [
        r"\bfracture\b",
        r"\btibial plateau fracture\b",
        r"\bosteochondral fracture\b",
        r"\binsufficiency fracture\b",
    ]
}   

In [380]:
def find_medical_findings(medical_terms, report):
    """
    Converts detected medical terms into the 12 original dataset labels.

    Original dataset labels:
        ACL
        MCL
        Medial Meniscus
        Lateral Meniscus
        Medial OA
        Lateral OA
        PF OA
        Effusion
        Synovitis
        Baker's
        Contusion
        Fracture

    Only non-negated medical terms are considered.
    The full translated report is also used as supporting context.
    """

    findings = set()

    # ---------------------------------------------------------
    # FULL REPORT
    # ---------------------------------------------------------

    report = str(report).lower()

    # ---------------------------------------------------------
    # KEYWORD GROUPS
    # ---------------------------------------------------------

    MENISCUS_ABNORMALITY = (
        "tear",
        "torn",
        "rupture",
        "degeneration",
        "degenerative",
        "meniscopathy",
        "lesion",
        "damage",
        "injury",
        "extrusion",
        "meniscal cyst",
    )

    OA_TERMS = (
        "osteoarthritis",
        "osteoarthrosis",
        "arthrosis",
        "chondromalacia",
        "cartilage degeneration",
        "cartilage loss",
        "cartilage thinning",
        "cartilage defect",
        "cartilage damage",
        "joint space narrowing",
        "chondropathy",
    )

    NORMAL_TERMS = (
        "normal",
        "intact",
        "unremarkable",
        "preserved",
        "within normal limits",
        "within normal limit",
        "no evidence",
        "without evidence",
        "no tear",
        "no tears",
        "without tear",
        "without tears",
        "no injury",
        "without injury",
        "no damage",
        "without damage",
        "no lesion",
        "without lesion",
        "no abnormality",
        "without abnormality",
    )

    # ---------------------------------------------------------
    # HELPER FUNCTIONS
    # ---------------------------------------------------------

    def has_any(text, keywords):
        return any(k in text for k in keywords)

    def is_normal_context(text):
        return has_any(text, NORMAL_TERMS)

    def anatomy_with_abnormality(text, anatomy_terms, abnormality_terms):
        """
        Checks whether an anatomical structure and an abnormality
        appear together, while avoiding obvious normal statements.
        """

        has_anatomy = has_any(text, anatomy_terms)
        has_abnormality = has_any(text, abnormality_terms)

        if not (has_anatomy and has_abnormality):
            return False

        # Extra safety:
        # do not classify text such as
        # "normal medial meniscus" or "medial meniscus without tear"
        if is_normal_context(text):
            return False

        return True

    # =========================================================
    # PROCESS DETECTED MEDICAL TERMS
    # =========================================================

    for item in medical_terms:

        # -----------------------------------------------------
        # Ignore explicitly negated terms
        # -----------------------------------------------------

        if item.get("negation", False):
            continue

        found = str(item.get("found", "")).lower()
        normalized = str(item.get("normalized", "")).lower()
        context = str(item.get("context", "")).lower()

        text = f"{context} {found} {normalized}"

        # =====================================================
        # 1. ACL
        # =====================================================

        if found in ("acl", "anterior cruciate ligament"):
            if not is_normal_context(text):
                findings.add("ACL")

        elif has_any(
            text,
            (
                "acl tear",
                "acl rupture",
                "acl injury",
                "acl sprain",
                "anterior cruciate ligament tear",
                "anterior cruciate ligament rupture",
                "anterior cruciate ligament injury",
            ),
        ):
            findings.add("ACL")

        # =====================================================
        # 2. MCL
        # =====================================================

        if found in (
            "mcl",
            "medial collateral ligament",
            "ligamento colateral medial",
            "ligamento colateral interno",
            "lcm",
        ):
            if not is_normal_context(text):
                findings.add("MCL")

        elif has_any(
            text,
            (
                "mcl tear",
                "mcl rupture",
                "mcl injury",
                "mcl sprain",
                "medial collateral ligament tear",
                "medial collateral ligament rupture",
                "medial collateral ligament injury",
            ),
        ):
            findings.add("MCL")

        # =====================================================
        # 3. MEDIAL MENISCUS
        # =====================================================

        medial_meniscus_terms = (
            "medial meniscus",
            "internal meniscus",
            "medial menisküs",
            "medijalni menisk",
            "medijalnog meniska",
        )

        if found in medial_meniscus_terms:

            # The medical-term detector found the anatomy,
            # but the surrounding context may contain the actual
            # pathology.
            if anatomy_with_abnormality(
                text,
                medial_meniscus_terms,
                MENISCUS_ABNORMALITY,
            ):
                findings.add("Medial Meniscus")

        # Direct pathology phrases are also checked.
        if has_any(
            text,
            (
                "medial meniscus tear",
                "medial meniscus rupture",
                "medial meniscus degeneration",
                "medial meniscus lesion",
                "medial meniscus damage",
                "medial meniscus injury",
                "medial meniscus extrusion",
                "medial meniscus meniscopathy",
            ),
        ):
            if not is_normal_context(text):
                findings.add("Medial Meniscus")

        # =====================================================
        # 4. LATERAL MENISCUS
        # =====================================================

        lateral_meniscus_terms = (
            "lateral meniscus",
            "lateral menisküs",
            "lateralni menisk",
            "lateralnog meniska",
        )

        if found in lateral_meniscus_terms:

            if anatomy_with_abnormality(
                text,
                lateral_meniscus_terms,
                MENISCUS_ABNORMALITY,
            ):
                findings.add("Lateral Meniscus")

        if has_any(
            text,
            (
                "lateral meniscus tear",
                "lateral meniscus rupture",
                "lateral meniscus degeneration",
                "lateral meniscus lesion",
                "lateral meniscus damage",
                "lateral meniscus injury",
                "lateral meniscus extrusion",
                "lateral meniscus meniscopathy",
            ),
        ):
            if not is_normal_context(text):
                findings.add("Lateral Meniscus")

        # =====================================================
        # 5. MEDIAL OA
        # =====================================================

        if anatomy_with_abnormality(
            text,
            (
                "medial compartment",
                "medial femorotibial",
                "medial tibiofemoral",
                "medial femoral condyle",
            ),
            OA_TERMS,
        ):
            findings.add("Medial OA")

        # Explicit medial OA phrases
        if has_any(
            text,
            (
                "medial compartment osteoarthritis",
                "medial femorotibial osteoarthritis",
                "medial tibiofemoral osteoarthritis",
                "medial compartment chondromalacia",
                "medial femorotibial chondromalacia",
            ),
        ):
            findings.add("Medial OA")

        # =====================================================
        # 6. LATERAL OA
        # =====================================================

        if anatomy_with_abnormality(
            text,
            (
                "lateral compartment",
                "lateral femorotibial",
                "lateral tibiofemoral",
                "lateral femoral condyle",
                "lateral tibial plateau",
            ),
            OA_TERMS,
        ):
            findings.add("Lateral OA")

        if has_any(
            text,
            (
                "lateral compartment osteoarthritis",
                "lateral femorotibial osteoarthritis",
                "lateral tibiofemoral osteoarthritis",
                "lateral compartment chondromalacia",
                "lateral femorotibial chondromalacia",
            ),
        ):
            findings.add("Lateral OA")

        # =====================================================
        # 7. PF OA
        # =====================================================

        if anatomy_with_abnormality(
            text,
            (
                "patellofemoral",
                "patella",
                "patellar",
                "femoropatellar",
            ),
            OA_TERMS,
        ):
            findings.add("PF OA")

        if has_any(
            text,
            (
                "patellofemoral osteoarthritis",
                "patellofemoral chondromalacia",
                "patellar chondromalacia",
                "patellofemoral joint degeneration",
            ),
        ):
            if not is_normal_context(text):
                findings.add("PF OA")

        # =====================================================
        # 8. EFFUSION
        # =====================================================

        if has_any(
            text,
            (
                "joint effusion",
                "knee joint effusion",
                "minimal joint effusion",
                "mild joint effusion",
                "hemarthrosis",
                "increased fluid",
                "fluid increase",
                "increased amount of fluid",
                "increased amount of intra-articular fluid",
            ),
        ):
            findings.add("Effusion")

        # =====================================================
        # 9. SYNOVITIS
        # =====================================================

        if has_any(
            text,
            (
                "synovitis",
                "knee synovitis",
                "synovial hypertrophy",
                "synovial thickening",
                "synovial proliferation",
            ),
        ):
            findings.add("Synovitis")

        # =====================================================
        # 10. BAKER'S
        # =====================================================

        if has_any(
            text,
            (
                "baker's cyst",
                "baker cyst",
                "baker’s cyst",
                "popliteal cyst",
                "popliteal cystic lesion",
            ),
        ):
            findings.add("Baker's")

        # =====================================================
        # 11. CONTUSION
        # =====================================================

        if has_any(
            text,
            (
                "bone contusion",
                "bone bruise",
                "muscle contusion",
                "bony contusion",
                "osseous contusion",
            ),
        ):
            findings.add("Contusion")

        # =====================================================
        # 12. FRACTURE
        # =====================================================

        if "fracture" in text:
            findings.add("Fracture")

    # =========================================================
    # IMPORTANT: FULL REPORT FALLBACK
    # =========================================================
    #
    # Some reports contain pathology that was not captured
    # correctly as a MedicalTerm. We therefore check explicit
    # pathology phrases in the complete translated report.
    #
    # We only use phrases that are relatively safe and specific.
    # =========================================================

    # ---------------------------------------------------------
    # MENISCUS
    # ---------------------------------------------------------

    if has_any(
        report,
        (
            "medial meniscus tear",
            "medial meniscus rupture",
            "medial meniscus degeneration",
            "medial meniscus lesion",
            "medial meniscus damage",
            "medial meniscus injury",
        ),
    ):
        findings.add("Medial Meniscus")

    if has_any(
        report,
        (
            "lateral meniscus tear",
            "lateral meniscus rupture",
            "lateral meniscus degeneration",
            "lateral meniscus lesion",
            "lateral meniscus damage",
            "lateral meniscus injury",
        ),
    ):
        findings.add("Lateral Meniscus")

    # ---------------------------------------------------------
    # LIGAMENTS
    # ---------------------------------------------------------

    if has_any(
        report,
        (
            "acl tear",
            "acl rupture",
            "acl injury",
            "anterior cruciate ligament tear",
            "anterior cruciate ligament rupture",
        ),
    ):
        findings.add("ACL")

    if has_any(
        report,
        (
            "mcl tear",
            "mcl rupture",
            "mcl injury",
            "medial collateral ligament tear",
            "medial collateral ligament rupture",
        ),
    ):
        findings.add("MCL")

    # ---------------------------------------------------------
    # EFFUSION
    # ---------------------------------------------------------

    if has_any(
        report,
        (
            "joint effusion",
            "knee joint effusion",
            "hemarthrosis",
            "increased fluid",
            "increased amount of fluid",
        ),
    ):
        # Do not add if the complete report clearly says
        # that there is no effusion.
        if not has_any(
            report,
            (
                "no joint effusion",
                "no evidence of knee effusion",
                "without joint effusion",
                "no effusion",
            ),
        ):
            findings.add("Effusion")

    # ---------------------------------------------------------
    # SYNOVITIS
    # ---------------------------------------------------------

    if has_any(
        report,
        (
            "synovitis",
            "synovial hypertrophy",
            "synovial thickening",
            "synovial proliferation",
        ),
    ):
        findings.add("Synovitis")

    # ---------------------------------------------------------
    # BAKER'S
    # ---------------------------------------------------------

    if has_any(
        report,
        (
            "baker's cyst",
            "baker cyst",
            "baker’s cyst",
            "popliteal cyst",
        ),
    ):
        if not has_any(
            report,
            (
                "no baker's cyst",
                "no baker cyst",
                "no popliteal cyst",
            ),
        ):
            findings.add("Baker's")

    # ---------------------------------------------------------
    # CONTUSION
    # ---------------------------------------------------------

    if has_any(
        report,
        (
            "bone contusion",
            "bone bruise",
            "bony contusion",
            "osseous contusion",
        ),
    ):
        if not has_any(
            report,
            (
                "no bone contusion",
                "no bone bruise",
                "without bone contusion",
            ),
        ):
            findings.add("Contusion")

    # ---------------------------------------------------------
    # FRACTURE
    # ---------------------------------------------------------

    if "fracture" in report:
        if not has_any(
            report,
            (
                "no fracture",
                "no acute fracture",
                "without fracture",
                "fractures: none",
            ),
        ):
            findings.add("Fracture")

    # ---------------------------------------------------------
    # FINAL OUTPUT
    # ---------------------------------------------------------

    return sorted(findings)




**Translate to english**

In [22]:
def translate_to_english(text, source_language):
    """
    Translates a medical report into English using Google Translate.

    Parameters
    ----------
    text : str
        Original medical report.

    source_language : str
        ISO 639-1 language code detected from the source report.

    Returns
    -------
    str
        Medical report translated into English.
    """

    if not isinstance(text, str) or not text.strip():
        return ""

    if not isinstance(source_language, str) or source_language == "unknown":
        return text

    if source_language == "en":
        return text

    translator = GoogleTranslator(
        source=source_language,
        target="en"
    )

    return translator.translate(text)

In [23]:
# RESTORE MEDICAL TERMS

def restore_terms(translated_text, placeholders):
    """
    Restores protected medical and technical terms after translation.

    The function tolerates minor formatting changes introduced
    by the translation service.
    """

    for token, term in placeholders.items():

        token_number = re.search(r"\d+", token)

        if not token_number:
            continue

        number = token_number.group()

        pattern = re.compile(
            r"x\s*term\s*" +
            re.escape(number) +
            r"\s*x",
            re.IGNORECASE
        )

        translated_text = pattern.sub(
            term,
            translated_text
        )

    return translated_text

In [24]:
# TRANSLATION VALIDATION

def validate_translation(
    translated_text,
    restored_text,
    placeholders
):
    """
    Performs basic validation after translation and restoration.

    The goal is to detect obvious translation and restoration
    problems, not to guarantee medical correctness.
    """

    warnings = []

    # Check translated text

    if not translated_text or not translated_text.strip():

        warnings.append(
            "Translation returned empty text."
        )

    # Check restored text

    if not restored_text or not restored_text.strip():

        warnings.append(
            "Restored text is empty."
        )

    # Check for remaining placeholders

    remaining_placeholders = re.findall(
        r"x\s*term\s*\d+\s*x",
        restored_text,
        re.IGNORECASE
    )

    if remaining_placeholders:

        warnings.append(
            "Unrestored placeholders detected: "
            + str(remaining_placeholders)
        )

    # Check that protected medical terms were restored

    for token, medical_term in placeholders.items():

        if medical_term not in restored_text:

            warnings.append(
                f"Medical term may have been lost: "
                f"{medical_term}"
            )

    return {
        "is_valid": len(warnings) == 0,
        "warnings": warnings
    }

In [25]:
# MEDICAL TERM NORMALIZATION

def normalize_medical_terms(text, glossary):
    """
    Normalizes medical terminology after translation.

    Longer expressions are processed first to prevent
    partial replacements.
    """

    sorted_terms = sorted(
        glossary.keys(),
        key=len,
        reverse=True
    )

    for term in sorted_terms:

        normalized_term = glossary[term]

        # Skip terms that are already normalized
        if term.lower() == normalized_term.lower():
            continue

        # Case-insensitive matching
        pattern = re.compile(
            r"(?<!\w)" +
            re.escape(term) +
            r"(?!\w)",
            re.IGNORECASE
        )

        text = pattern.sub(
            normalized_term,
            text
        )

    return text

In [26]:
# TRANSLATION PIPELINE TEST

# 1. Detect source language

source_language = detect_language(text)

print("\n--- SOURCE LANGUAGE ---")
print(source_language)


# 2. Protect medical and MRI technical terms

protected_text, placeholders, flagged_terms = protect_terms(
    text,
    medical_dictionary,
    MRI_TECHNICAL_TERMS
)

print("\n--- PROTECTED TEXT ---")
print(protected_text)

print("\n--- PROTECTED TERMS ---")
print(placeholders)

print("\n--- FLAGGED TERMS ---")
print(flagged_terms)


# 3. Translate protected text into English

translated_text = translate_to_english(
    protected_text,
    source_language
)

print("\n--- TRANSLATED TEXT ---")
print(translated_text)


# 4. Restore protected medical terms

restored_text = restore_terms(
    translated_text,
    placeholders
)

print("\n--- RESTORED TEXT ---")
print(restored_text)


# 5. Validate translation and restoration

validation = validate_translation(
    translated_text,
    restored_text,
    placeholders
)

print("\n--- TRANSLATION VALIDATION ---")
print("Valid:", validation["is_valid"])

if validation["warnings"]:

    print("Warnings:")

    for warning in validation["warnings"]:
        print("-", warning)


# 6. Normalize medical terminology

normalized_text = normalize_medical_terms(
    restored_text,
    medical_dictionary
)

print("\n--- NORMALIZED TEXT ---")
print(normalized_text)


# 7. Validate final medical content

medical_validation = validate_medical_content(
    normalized_text
)

print("\n--- MEDICAL CONTENT VALIDATION ---")
print("Valid:", medical_validation["is_valid"])

if medical_validation["warnings"]:

    print("Warnings:")

    for warning in medical_validation["warnings"]:
        print("-", warning)


--- SOURCE LANGUAGE ---
en

--- PROTECTED TEXT ---
Exam Type: XTERM224X KNEE LEFT WO CONTRAST
Exam Date and Time: [DATE] [TIME]
Indication: Left knee pain, suspected medial XTERM182X.
Comparison: [REDACTED].

TECHNIQUE:
XTERM224X of the left     knee was performed on a 1.5 Tesla system with a dedicated knee coil in three planes (axial, sagittal, and coronal), using a standard non-contrast protocol.

FINDINGS:
OSSEOUS STRUCTURES:
Alignment is anatomic. No acute fracture or XTERM197X. Background marrow signal is normal without infiltrative marrow process. No aggressive osseous lesions.

JOINT SPACE:
Physiologic joint fluid without large volume effusion. No intra-articular bodies. No XTERM187X.

MEDIAL COMPARTMENT:
XTERM167X: Normal in morphology and signal.
Medial compartment cartilage: No focal full-thickness cartilage defects.

LATERAL COMPARTMENT:
XTERM160X: Normal in morphology and signal.
Lateral compartment cartilage: No focal full-thickness cartilage defects.

PATELLOFEMORAL COMP

In [30]:
# TRANSLATION PIPELINE

def process_translation(row):
    """
    Processes one medical report through the translation pipeline.

    The report is:
    1. Checked for its source language.
    2. Protected from unwanted translation of medical terms.
    3. Translated into English.
    4. Restored with the protected terminology.
    5. Normalized using the medical dictionary.

    If translation fails, the original report is returned
    instead of stopping the entire dataset processing.
    """

    text = row["Report"]
    source_language = row["ReportLanguage"]

    if not isinstance(text, str) or not text.strip():
        return ""

    try:

        # Protect medical and technical terminology
        protected_text, placeholders, flagged_terms = protect_terms(
            text,
            medical_dictionary,
            MRI_TECHNICAL_TERMS
        )

        # Translate the report into English
        translated_text = translate_to_english(
            protected_text,
            source_language
        )

        # Restore protected terminology
        restored_text = restore_terms(
            translated_text,
            placeholders
        )

        # Normalize medical terminology
        normalized_text = normalize_medical_terms(
            restored_text,
            medical_dictionary
        )

        return normalized_text

    except Exception as error:

        print(
            f"Translation failed for "
            f"StudyInstanceUID {row['StudyInstanceUID']}: "
            f"{error}"
        )

        return text

In [31]:
# CREATE TRANSLATED REPORT COLUMN

df["TranslatedReport"] = df.apply(
    process_translation,
    axis=1
)

In [33]:
df["TranslatedReport"]

0       Technique: MRI of the knee. Results: Internal ...
1       [DATE]: * MR Knee Right 15ch AA Clinical Infor...
2       Findings:\nThere are no significant alteration...
3        In the medial compartment, the meniscus is no...
4       FINDINGS:\n\nFractures:\nNone.\n\nJoint alignm...
                              ...                        
4402    Exam Type: MRI KNEE RIGHT WO CONTRAST\nExam Da...
4403    [DATE]: *MR Knee Right 15ch AA Clinical Inform...
4404    RIGHT KNEE MRG. Examination protocol: Multipla...
4405    MRI of Knee with \n-Locator, SG PD FatSat, SG ...
4406    Technique: MRI of the knee. Results: patellar ...
Name: TranslatedReport, Length: 4407, dtype: object

In [34]:
df["TranslatedReport"].isna().sum()

0

In [35]:
# CHECK NON-ENGLISH TRANSLATIONS

df[
    df["ReportLanguage"] != "en"
][
    [
        "Report",
        "ReportLanguage",
        "TranslatedReport"
    ]
].head(10)

,Report,ReportLanguage,TranslatedReport
0,Técnica: RMN de la rodilla. Resultados: Rotura...,es,Technique: MRI of the knee. Results: Internal ...
1,[DATE]: * MR Knie Rechts 15ch AA Klinische Inl...,nl,[DATE]: * MR Knee Right 15ch AA Clinical Infor...
2,Hallazgos:\nNo hay alteraciones en significati...,es,Findings:\nThere are no significant alteration...
4,CONSTATATIONS :\n\nFractures :\nAucune.\n\nAli...,fr,FINDINGS:\n\nFractures:\nNone.\n\nJoint alignm...
5,Antecedentes Clínicos:\nCondromalacia rotulian...,es,Clinical History:\nChondromalacia patella.\nFi...
7,"МР находка: МР данни за ставен излив. Костите,...",bg,MR finding: MR evidence of joint effusion. The...
8,"SOL DİZ MRG. Tetkik protokolü: Çok düzlemli, ç...",tr,LEFT KNEE MRG. Examination protocol: Multiplan...
11,Antecedentes Clínicos:\nSíndrome meniscal rodi...,es,Clinical History:\nRight knee meniscal syndrom...
13,MRI ΑΡΙΣΤΕΡΟΥ ΓΟΝΑΤΟΣ ΤΕΧΝΙΚΗ Η εξέταση έγινε ...,el,MRI LEFT KNEE TECHNIQUE The examination was pe...
15,MRI ΑΡΙΣΤΕΡΟΥ ΓΟΝΑΤΟΣ ΤΕΧΝΙΚΗ Η εξέταση έγινε ...,el,MRI LEFT KNEE TECHNIQUE The examination was pe...


**NLP Model**

In [36]:
nlp = spacy.load("en_core_sci_sm")

In [37]:
text_for_scispacy = normalized_text
doc = nlp(text_for_scispacy)

**Tokenization and Lemmatizacion**

In [38]:
token_data = [
    {
        "text": token.text,
        "lemma": token.lemma_,
        "pos": token.pos_,
        "dep": token.dep_
    }
    for token in doc
]

lemmatized_tokens = [
    token["lemma"]
    for token in token_data
    if token["pos"] != "PUNCT"
]

print(lemmatized_tokens)

['exam', 'type', 'mri', 'KNEE', 'left', 'will', 'CONTRAST', '\n', 'Exam', 'Date', 'and', 'time', 'date', 'time', '\n', 'Indication', 'left', 'knee', 'pain', 'suspect', 'medial', 'meniscus', 'tear', '\n', 'comparison', 'redacted', '\n\n', 'technique', '\n', 'mri', 'of', 'the', 'left', '    ', 'knee', 'be', 'perform', 'on', 'a', '1.5', 'Tesla', 'system', 'with', 'a', 'dedicated', 'knee', 'coil', 'in', 'three', 'plane', 'axial', 'sagittal', 'and', 'coronal', 'use', 'a', 'standard', 'non-contrast', 'protocol', '\n\n', 'finding', '\n', 'osseous', 'structures', '\n', 'Alignment', 'be', 'anatomic', 'no', 'acute', 'fracture', 'or', 'bone', 'contusion', 'background', 'marrow', 'signal', 'be', 'normal', 'without', 'infiltrative', 'marrow', 'process', 'no', 'aggressive', 'osseous', 'lesion', '\n\n', 'joint', 'space', '\n', 'physiologic', 'joint', 'fluid', 'without', 'large', 'volume', 'effusion', 'no', 'intra-articular', 'body', 'no', 'Baker', "'s", 'cyst', '\n\n', 'medial', 'compartment', '\n', 

In [39]:
# CREATE TOKEN COLUMN

def tokenize_and_lemmatize(text):
    """
    Tokenizes and lemmatizes an English medical report.
    Punctuation is excluded from the final token list.
    """

    if not isinstance(text, str) or not text.strip():
        return []

    doc = nlp(text)

    return [
        token.lemma_
        for token in doc
        if not token.is_punct
    ]


df["Tokens"] = df["TranslatedReport"].apply(
    tokenize_and_lemmatize
)

In [40]:
# CHECK TOKENIZED REPORTS

df[
    [
        "Report",
        "ReportLanguage",
        "TranslatedReport",
        "Tokens"
    ]
].head()

,Report,ReportLanguage,TranslatedReport,Tokens
0,Técnica: RMN de la rodilla. Resultados: Rotura...,es,Technique: MRI of the knee. Results: Internal ...,"[technique, mri, of, the, knee, result, intern..."
1,[DATE]: * MR Knie Rechts 15ch AA Klinische Inl...,nl,[DATE]: * MR Knee Right 15ch AA Clinical Infor...,"[date, MR, Knee, Right, 15ch, AA, Clinical, In..."
2,Hallazgos:\nNo hay alteraciones en significati...,es,Findings:\nThere are no significant alteration...,"[finding, \n, there, be, no, significant, alte..."
3,"In the medial compartment, the meniscus is no...",en,"In the medial compartment, the meniscus is no...","[ , in, the, medial, compartment, the, meniscu..."
4,CONSTATATIONS :\n\nFractures :\nAucune.\n\nAli...,fr,FINDINGS:\n\nFractures:\nNone.\n\nJoint alignm...,"[finding, \n\n, fracture, \n, none, \n\n, join..."


**Medical Terms Extraction**

In [381]:
# REPORT SECTION IDENTIFICATION

def get_section(text, position):
    """
    Identifies the report section containing a medical finding.
    """

    before_text = text[:position].lower()

    if "impression:" in before_text:
        return "Impression"

    elif "findings:" in before_text:
        return "Findings"

    elif "results:" in before_text:
        return "Results"

    elif "technique:" in before_text:
        return "Technique"

    return "Unknown"

In [382]:
# MEDICAL FINDING EXTRACTION

def find_medical_terms(text, medical_dictionary):
    """
    Identifies medical terms in an English medical report,
    normalizes them, identifies their report section,
    and stores local context around each term.
    """

    if not isinstance(text, str) or not text.strip():
        return []

    medical_terms_found = []

    for term, normalized_term in medical_dictionary.items():

        pattern = re.compile(
            r'(?<!\w)' + re.escape(term) + r'(?!\w)',
            re.IGNORECASE
        )

        for match in pattern.finditer(text):

            start = match.start()
            end = match.end()

            # Get local context around the medical term
            context_start = max(0, start - 50)
            context_end = min(len(text), end + 50)

            context = text[
                context_start:context_end
            ]

            medical_terms_found.append({
                "found": match.group(),
                "normalized": normalized_term,
                "start": start,
                "end": end,
                "context": context,
                "section": get_section(
                    text,
                    start
                )
            })

    return medical_terms_found

In [383]:
# CREATE MEDICAL TERMS COLUMN

df["MedicalTerms"] = df["TranslatedReport"].apply(
    lambda text: find_medical_terms(
        text,
        medical_dictionary
    )
)

In [384]:
# CHECK MEDICAL TERMS

df[
    [
        "Report",
        "ReportLanguage",
        "TranslatedReport",
        "Tokens",
        "MedicalTerms"
    ]
].head()

,Report,ReportLanguage,TranslatedReport,Tokens,MedicalTerms
0,Técnica: RMN de la rodilla. Resultados: Rotura...,es,Technique: MRI of the knee. Results: Internal ...,"[technique, mri, of, the, knee, result, intern...","[{'found': 'meniscus tear', 'normalized': 'men..."
1,[DATE]: * MR Knie Rechts 15ch AA Klinische Inl...,nl,[DATE]: * MR Knee Right 15ch AA Clinical Infor...,"[date, MR, Knee, Right, 15ch, AA, Clinical, In...","[{'found': 'meniscus tear', 'normalized': 'men..."
2,Hallazgos:\nNo hay alteraciones en significati...,es,Findings:\nThere are no significant alteration...,"[finding, \n, there, be, no, significant, alte...","[{'found': 'medial meniscus', 'normalized': 'm..."
3,"In the medial compartment, the meniscus is no...",en,"In the medial compartment, the meniscus is no...","[ , in, the, medial, compartment, the, meniscu...","[{'found': 'MCL', 'normalized': 'MCL', 'start'..."
4,CONSTATATIONS :\n\nFractures :\nAucune.\n\nAli...,fr,FINDINGS:\n\nFractures:\nNone.\n\nJoint alignm...,"[finding, \n\n, fracture, \n, none, \n\n, join...","[{'found': 'ACL', 'normalized': 'ACL', 'start'..."


**Search for Negation**

In [385]:
# NEGATION ANALYSIS

NEGATION_TERMS = [
    "no",
    "not",
    "without",
    "none",
    "absent",
    "absence",
    "negative",
    "free of",
    "normal",
    "intact",
    "preserved",
    "unremarkable",
    "no evidence of",
    "no signs of",
    "no significant"
]

for term in NEGATION_TERMS:

    count = df["TranslatedReport"].str.contains(
        rf"\b{re.escape(term)}\b",
        case=False,
        na=False
    ).sum()

    print(f"{term}: {count}")

no: 3254
not: 868
without: 1592
none: 195
absent: 20
absence: 14
negative: 4
free of: 2
normal: 3057
intact: 1471
preserved: 912
unremarkable: 317
no evidence of: 556
no signs of: 221
no significant: 647


**Validate Negation**

In [386]:
# REVIEW NEGATION

pattern = r"\b(no|not|without|absent|absence|negative|normal|intact|preserved|unremarkable)\b"

negation_reports = df[
    df["TranslatedReport"].str.contains(
        pattern,
        case=False,
        na=False,
        regex=True
    )
]["TranslatedReport"]

print(negation_reports.head(20).to_string(index=False))

[DATE]: * MR Knee Right 15ch AA Clinical Inform...
Findings:\nThere are no significant alterations...
 In the medial compartment, the meniscus is not...
FINDINGS:\n\nFractures:\nNone.\n\nJoint alignme...
Clinical History:\nChondromalacia patella.\nFin...
MRI of left knee with -Locator, SG PD FatSat, S...
MR finding: MR evidence of joint effusion. The ...
LEFT KNEE MRG. Examination protocol: Multiplana...
MRI of left Knee with \n-Locator, SG PD FatSat,...
Findings:  There is a mild knee effusion. There...
Clinical History:\nRight knee meniscal syndrome...
    The study reveals anterior tibial translati...
MRI LEFT KNEE TECHNIQUE The examination was per...
> MRI of knee was performed in sagittal, axial ...
MRI LEFT KNEE TECHNIQUE The examination was per...
On the performed sequences, severe knee osteoar...
MRI of right knee in sagittal section (Proton d...
 In the medial compartment, there is suspicious...
MRI of left Knee with \n-Locator, SG PD FatSat,...
TECHNIQUE: The examination was 

In [387]:
def detect_negation(text, start, window=80):
    """
    Detects negation or normality associated with a medical term.
    Checks both before and after the medical term.
    """

    # Context before the medical term
    before = text[
        max(0, start - window):start
    ].lower()

    # Context after the medical term
    after = text[
        start:min(len(text), start + window)
    ].lower()

    # Only use the current sentence
    before = re.split(r"[.!?\n]", before)[-1]
    after = re.split(r"[.!?\n]", after)[0]

    # -----------------------------------------
    # NEGATION BEFORE THE TERM
    # -----------------------------------------

    for term in NEGATION_TERMS:

        pattern = rf"\b{re.escape(term)}\b"

        if re.search(pattern, before):
            return True

    # -----------------------------------------
    # NORMALITY AFTER THE TERM
    # -----------------------------------------

    NORMAL_TERMS = [
        "normal",
        "normally",
        "preserved",
        "intact",
        "unremarkable",
        "without abnormalities",
        "without abnormality",
        "without alteration",
        "without alterations",
        "no abnormality",
        "no abnormalities",
        "no evidence of"
    ]

    for term in NORMAL_TERMS:

        pattern = rf"\b{re.escape(term)}\b"

        if re.search(pattern, after):
            return True

    return False

**Negation Detection**

In [388]:
def add_negation_to_terms(row):

    text = row["TranslatedReport"]
    terms = row["MedicalTerms"]

    for item in terms:

        item["negation"] = detect_negation(
            text,
            item["start"]
        )

    return terms


df["MedicalTerms"] = df.apply(
    add_negation_to_terms,
    axis=1
)

**Check Negation**

In [389]:

print("Columns:")
print(df.columns.tolist())

print("\n--- DATASET PREVIEW ---")

print(
    df[
        [
            "Report",
            "ReportLanguage",
            "TranslatedReport",
            "MedicalTerms"
        ]
    ].head(5).to_string(index=False)
)


print("\n--- MEDICAL TERMS + NEGATION ---")

for i, terms in enumerate(df["MedicalTerms"].head(5)):

    print(f"\nREPORT {i}")

    if not terms:
        print("No medical terms found.")
        continue

    for term in terms:

        print(
            f"Found: {term['found']} | "
            f"Normalized: {term['normalized']} | "
            f"Section: {term.get('section', 'Unknown')} | "
            f"Negation: {term.get('negation', False)}")

Columns:
['StudyInstanceUID', 'Report', 'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture', 'ReportLanguage', 'TranslatedReport', 'Tokens', 'MedicalTerms', 'MedicalFindings']

--- DATASET PREVIEW ---
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

**Check Current df**

In [390]:
display(df[
        [
            "Report",
            "ReportLanguage",
            "TranslatedReport",
            "MedicalTerms"
        ]
    ].head())

,Report,ReportLanguage,TranslatedReport,MedicalTerms
0,Técnica: RMN de la rodilla. Resultados: Rotura...,es,Technique: MRI of the knee. Results: Internal ...,"[{'found': 'meniscus tear', 'normalized': 'men..."
1,[DATE]: * MR Knie Rechts 15ch AA Klinische Inl...,nl,[DATE]: * MR Knee Right 15ch AA Clinical Infor...,"[{'found': 'meniscus tear', 'normalized': 'men..."
2,Hallazgos:\nNo hay alteraciones en significati...,es,Findings:\nThere are no significant alteration...,"[{'found': 'medial meniscus', 'normalized': 'm..."
3,"In the medial compartment, the meniscus is no...",en,"In the medial compartment, the meniscus is no...","[{'found': 'MCL', 'normalized': 'MCL', 'start'..."
4,CONSTATATIONS :\n\nFractures :\nAucune.\n\nAli...,fr,FINDINGS:\n\nFractures:\nNone.\n\nJoint alignm...,"[{'found': 'ACL', 'normalized': 'ACL', 'start'..."


**Medical Founded Terms to Dataset Labes**

In [391]:
df["MedicalFindings"] = df.apply(
    lambda row: find_medical_findings(
        row["MedicalTerms"],
        row["TranslatedReport"]
    ),
    axis=1
)

**Current State of df**

In [392]:
display(df[
        [
            "Report",
            "ReportLanguage",
            "TranslatedReport",
            "MedicalTerms",
            "MedicalFindings"
        ]
    ].head(10))

,Report,ReportLanguage,TranslatedReport,MedicalTerms,MedicalFindings
0,Técnica: RMN de la rodilla. Resultados: Rotura...,es,Technique: MRI of the knee. Results: Internal ...,"[{'found': 'meniscus tear', 'normalized': 'men...",[Medial OA]
1,[DATE]: * MR Knie Rechts 15ch AA Klinische Inl...,nl,[DATE]: * MR Knee Right 15ch AA Clinical Infor...,"[{'found': 'meniscus tear', 'normalized': 'men...",[]
2,Hallazgos:\nNo hay alteraciones en significati...,es,Findings:\nThere are no significant alteration...,"[{'found': 'medial meniscus', 'normalized': 'm...","[Baker's, Effusion, PF OA]"
3,"In the medial compartment, the meniscus is no...",en,"In the medial compartment, the meniscus is no...","[{'found': 'MCL', 'normalized': 'MCL', 'start'...",[Contusion]
4,CONSTATATIONS :\n\nFractures :\nAucune.\n\nAli...,fr,FINDINGS:\n\nFractures:\nNone.\n\nJoint alignm...,"[{'found': 'ACL', 'normalized': 'ACL', 'start'...","[Baker's, Effusion, Fracture, MCL]"
5,Antecedentes Clínicos:\nCondromalacia rotulian...,es,Clinical History:\nChondromalacia patella.\nFi...,"[{'found': 'Medial meniscus', 'normalized': 'm...",[PF OA]
6,"MRI of left knee with -Locator, SG PD FatSat, ...",en,"MRI of left knee with -Locator, SG PD FatSat, ...","[{'found': 'ACL', 'normalized': 'ACL', 'start'...","[ACL, Effusion]"
7,"МР находка: МР данни за ставен излив. Костите,...",bg,MR finding: MR evidence of joint effusion. The...,"[{'found': 'ACL', 'normalized': 'ACL', 'start'...","[ACL, Effusion, Medial Meniscus]"
8,"SOL DİZ MRG. Tetkik protokolü: Çok düzlemli, ç...",tr,LEFT KNEE MRG. Examination protocol: Multiplan...,"[{'found': 'ACL', 'normalized': 'ACL', 'start'...",[Effusion]
9,"MRI of left Knee with \n-Locator, SG PD FatSat...",en,"MRI of left Knee with \n-Locator, SG PD FatSat...","[{'found': 'ACL', 'normalized': 'ACL', 'start'...","[Contusion, Effusion, Fracture, Lateral Menisc..."


**Review Reports without detected findings**

In [393]:
# REVIEW REPORTS WITHOUT DETECTED FINDINGS

display(
    df[
        df["MedicalFindings"].apply(lambda x: len(x) == 0)
    ][
        [
            "Report",
            "ReportLanguage",
            "TranslatedReport",
            "MedicalTerms",
            "MedicalFindings"
        ]
    ].head(10)
)

,Report,ReportLanguage,TranslatedReport,MedicalTerms,MedicalFindings
1,[DATE]: * MR Knie Rechts 15ch AA Klinische Inl...,nl,[DATE]: * MR Knee Right 15ch AA Clinical Infor...,"[{'found': 'meniscus tear', 'normalized': 'men...",[]
13,MRI ΑΡΙΣΤΕΡΟΥ ΓΟΝΑΤΟΣ ΤΕΧΝΙΚΗ Η εξέταση έγινε ...,el,MRI LEFT KNEE TECHNIQUE The examination was pe...,"[{'found': 'MRI', 'normalized': 'MRI', 'start'...",[]
15,MRI ΑΡΙΣΤΕΡΟΥ ΓΟΝΑΤΟΣ ΤΕΧΝΙΚΗ Η εξέταση έγινε ...,el,MRI LEFT KNEE TECHNIQUE The examination was pe...,"[{'found': 'Wiberg type II patella', 'normaliz...",[]
24,No evidence of knee effusion. Normal medial an...,en,No evidence of knee effusion. Normal medial an...,"[{'found': 'ACL', 'normalized': 'ACL', 'start'...",[]
25,Diz eklemi içi sıvı miktarı hafif artmış. Çapr...,tr,The amount of fluid inside the knee joint has ...,"[{'found': 'lateral meniscus', 'normalized': '...",[]
28,REPORT: No joint effusion. No intra-articular...,en,REPORT: No joint effusion. No intra-articular...,"[{'found': 'joint effusion', 'normalized': 'jo...",[]
30,Diz eklemi içi sıvı miktarı normal. Çapraz ve ...,tr,The amount of fluid inside the knee joint is n...,"[{'found': 'lateral meniscus', 'normalized': '...",[]
44,Técnica: RMN de la rodilla. Resultados: Lesión...,es,Technique: MRI of the knee. Results: Osteochon...,"[{'found': 'Osteochondral injury', 'normalized...",[]
61,Técnica: RMN de la rodilla. Resultados: Rotura...,es,Technique: MRI of the knee. Results: Internal ...,"[{'found': 'meniscus tear', 'normalized': 'men...",[]
66,[DATE]: MR Knie Rechts 15ch AA: Klinische inli...,nl,[DATE]: MR Knee Right 15ch AA: Clinical inform...,"[{'found': 'bone marrow edema', 'normalized': ...",[]


In [394]:
# REPORTS WITHOUT MEDICAL FINDINGS

empty_findings = df[
    df["MedicalFindings"].apply(len) == 0
]

print("Total:", len(empty_findings))

Total: 624


In [398]:
from collections import Counter

empty_term_counts = Counter()

for terms in empty_findings["MedicalTerms"]:

    for item in terms:

        if not item.get("negation", False):

            empty_term_counts[item["normalized"]] += 1


print("Medical terms not ruled out in reports with no findings:\n")

for term, count in empty_term_counts.most_common():

    print(f"{count:4}  {term}")

Medical terms not ruled out in reports with no findings:

 203  MRI
  96  Hoffa Fat Pad Impingement
  88  medial meniscus
  66  FS
  46  meniscus tear
  39  synovial cyst
  39  osteoarthritis
  28  lateral meniscus
  27  ACL
  23  PCL
  20  tendinosis
  18  tendinopathy
  16  bone marrow edema
  15  ganglion cyst
   9  osteochondral injury
   9  MCL
   8  osteochondral lesion
   8  soft tissue edema
   8  LCL
   7  parameniscal cyst
   6  Wiberg type II patella
   6  chondromalacia
   6  MPFL
   6  patella alta
   5  osteochondral loose body
   5  loose body
   5  patellar maltracking
   5  cartilage loss
   5  posterior horn of medial meniscus
   5  knee osteoarthritis
   4  quadriceps tendinosis
   4  subcutaneous edema
   3  lipoma arborescens
   3  iliotibial band syndrome
   3  infrapatellar bursitis
   2  cartilage defect
   2  ACL mucoid degeneration
   2  osteochondroma
   2  patellar tendon impingement
   2  PCL sprain
   2  patellofemoral syndrome
   2  cartilage thinning
   